---
short_title: Manuelle Analyse
numbering:
    heading_1: true
    heading_2: true
    title: true
---

# Manuelle Analyse: beschreibende Statistik

Nun haben wir alle Bausteine zusammen: Dateien einlesen, Felder typisieren und
Daten in Listen und Dictionaries strukturieren. In diesem Kapitel beantworten
wir Fragen an den Datensatz **mit reinen Python-Bordmitteln** – ohne {term}`Pandas`.

:::{seealso} Vorwissen
:icon: false

Dieses Kapitel setzt
[Datenstrukturen](005-Datenstrukturen.ipynb) und die
[erweiterte CSV-Verarbeitung](010-Erweiterte_CSV_Verarbeitung.ipynb)
voraus.
:::

## Den Datensatz einlesen

Wir lesen die Datei in eine Liste typisierter Dictionaries ein. Die
Hilfsfunktionen `finde_datensatz()` und `lade_daten()` liegen als
`csv_helfer.py` im Projekt und werden hier importiert. Diese „Liste von
Dictionaries“ ist unsere kleine Datenbank in Python.


In [1]:
import sys
sys.path.append(".")          # erlaubt den Import des Helfers csv_helfer.py

from collections import Counter, defaultdict
from csv_helfer import finde_datensatz, lade_daten

DATEN = lade_daten(finde_datensatz())
print("Anzahl Datensätze:", len(DATEN))
print("Erster Datensatz:", DATEN[0])


Anzahl Datensätze: 180
Erster Datensatz: {'isbn': '9780241951446', 'author': 'Murakami, Haruki', 'year': 1984, 'title': 'Norwegian Wood', 'sales_year': 2020, 'sales': 61}


In [2]:
# Die zu analysierende Spalte als einfache Liste herausziehen
verkaufszahlen = [satz["sales"] for satz in DATEN]
print("Erste zehn Werte:", verkaufszahlen[:10], "...")

Erste zehn Werte: [61, 40, 15, 15, 14, 14, 8, 8, 8, 8] ...


## Beschreibende Statistik von Hand

Wir beginnen mit den einfachen Kennzahlen und bauen uns Schritt für Schritt
unsere eigenen Funktionen. So wird transparent, was die Zahlen bedeuten.

### Anzahl und Summe

`len()` und `sum()` gehören zum eingebauten Werkzeugkasten von Python.

In [3]:
print("Anzahl:", len(verkaufszahlen))
print("Summe: ", sum(verkaufszahlen))

Anzahl: 180
Summe:  2760


### Minimum und Maximum

Auch `min()` und `max()` sind eingebaut. Wollen wir wissen, *welches Buch* den
höchsten Wert hat, sortieren bzw. durchsuchen wir die Liste von Dictionaries.

In [4]:
print("Minimum:", min(verkaufszahlen))
print("Maximum:", max(verkaufszahlen))

# Zu einem Extremwert den zugehörigen Datensatz finden
bestseller = max(DATEN, key=lambda satz: satz["sales"])
print("Bestseller:", bestseller["title"], "-", bestseller["sales"], "Verkäufe")

Minimum: 5
Maximum: 151
Bestseller: Der Prozess - 151 Verkäufe


### Mittelwert

Der **Mittelwert** (arithmetisches Mittel) ist die Summe geteilt durch die
Anzahl. Er reagiert empfindlich auf Ausreißer – genau deshalb schauen wir uns
zusätzlich den Median an.

In [5]:
def mittelwert(werte):
    """Arithmetisches Mittel: Summe geteilt durch Anzahl."""
    return sum(werte) / len(werte)

print(f"Mittelwert: {mittelwert(verkaufszahlen):.2f}")

Mittelwert: 15.33


### Median

Der **Median** ist der mittlere Wert der der Größe nach sortierten Liste. Bei
einer geraden Anzahl gibt es zwei mittlere Werte; dann wird deren Mittelwert
gebildet. Der Median ist robuster gegenüber Ausreißern als der Mittelwert.

In [6]:
def median(werte):
    """Median: der mittlere Wert der sortierten Liste."""
    sortiert = sorted(werte)
    n = len(sortiert)
    mitte = n // 2
    if n % 2 == 1:                      # ungerade Anzahl
        return sortiert[mitte]
    return (sortiert[mitte - 1] + sortiert[mitte]) / 2

print(f"Median: {median(verkaufszahlen)}")

Median: 8.0


### Modus

Der **Modus** ist der häufigste Wert. Treten mehrere Werte gleich häufig auf,
wählen wir willkürlich den kleinsten. {term}`Counter`.most_common()` macht die Arbeit
angenehm kurz.

In [7]:
def modus(werte):
    """Häufigster Wert (Modus); bei Gleichstand der kleinere Wert."""
    zaehler = Counter(werte)
    haeufigster = max(zaehler.items(), key=lambda paar: (paar[1], -paar[0]))
    return haeufigster[0], haeufigster[1]

wert, anzahl = modus(verkaufszahlen)
print(f"Modus: {wert} (kommt {anzahl}-mal vor)")

Modus: 6 (kommt 37-mal vor)


### Standardabweichung

Die **Standardabweichung** beschreibt, wie stark die Werte um den Mittelwert
streuen. Zuerst das Mittel, dann die mittlere quadratische Abweichung
(**Varianz**), daraus die Wurzel:

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \qquad
s = \sqrt{\frac{1}{n-1}\sum_{i=1}^{n}(x_i-\bar{x})^2}$$

Die Variante mit `n-1` heißt **Stichproben**-Standardabweichung, die mit `n`
**Grundgesamtheits**-Standardabweichung. Im Kurs berechnen wir beide.

In [8]:
def standardabweichung(werte, stichprobe=False):
    """Standardabweichung; stichprobe=True verwendet den Nenner n-1."""
    n = len(werte)
    my = mittelwert(werte)
    quadratsumme = sum((x - my) ** 2 for x in werte)
    nenner = n - 1 if stichprobe else n
    return (quadratsumme / nenner) ** 0.5

print(f"Grundgesamtheit (n):   {standardabweichung(verkaufszahlen):.2f}")
print(f"Stichprobe (n-1):      {standardabweichung(verkaufszahlen, stichprobe=True):.2f}")

Grundgesamtheit (n):   23.32
Stichprobe (n-1):      23.38


## Gruppieren und Aggregieren

Meist reicht eine Kennzahl für den gesamten Datensatz nicht. Wir wollen
**gruppieren** – etwa die Verkäufe pro Jahr – und je Gruppe **aggregieren**
(summe, Anzahl usw.). Das Muster ist immer ähnlich:

1. Ein leeres Dictionary anlegen.
2. Über alle Datensätze laufen.
3. Für jeden Datensatz die Gruppe als Schlüssel nutzen und den Wert
   aufsummieren bzw. anhängen.

### Verkäufe pro Jahr

In [9]:
verkauf_pro_jahr = {}
for satz in DATEN:
    jahr = satz["sales_year"]
    verkauf_pro_jahr[jahr] = verkauf_pro_jahr.get(jahr, 0) + satz["sales"]

for jahr in sorted(verkauf_pro_jahr):
    print(f"{jahr}: {verkauf_pro_jahr[jahr]:>5} Verkäufe")

2020:   319 Verkäufe
2021:   454 Verkäufe
2022:   561 Verkäufe
2023:   388 Verkäufe
2024:   601 Verkäufe
2025:   437 Verkäufe


### Top-Autor\*innen

Wollen wir die Verkäufe nach Autor\*in gruppieren, ist ein `defaultdict(int)`
angenehm: Fehlende Schlüssel starten automatisch bei `0`.

In [10]:
verkauf_pro_autor = defaultdict(int)
for satz in DATEN:
    verkauf_pro_autor[satz["author"]] += satz["sales"]

# nach Summe absteigend sortieren
ranking = sorted(verkauf_pro_autor.items(),
                 key=lambda paar: paar[1], reverse=True)

for platz, (autor, summe) in enumerate(ranking[:5], start=1):
    print(f"{platz}. {autor}: {summe} Verkäufe")

1. Kafka, Franz: 252 Verkäufe
2. Murakami, Haruki: 225 Verkäufe
3. Atwood, Margaret: 213 Verkäufe
4. Mbembe, Achille: 202 Verkäufe
5. Han, Byung-Chul: 174 Verkäufe


### Top-Titel

In [11]:
verkauf_pro_titel = defaultdict(int)
for satz in DATEN:
    verkauf_pro_titel[satz["title"]] += satz["sales"]

top_titel = sorted(verkauf_pro_titel.items(),
                   key=lambda paar: paar[1], reverse=True)[:5]
for titel, summe in top_titel:
    print(f"{summe:>4}  {titel}")

 252  Der Prozess
 225  Norwegian Wood
 213  The Handmaid's Tale
 202  Politik der Feindschaft
 174  Die Austreibung des Anderen


## Alles zusammenfassen

Zum Schluss bündeln wir unsere Funktionen in einer einzigen Ausgabe. So sieht
man auf einen Blick, dass Mittelwert und Median weit auseinanderliegen – ein
typisches Zeichen für eine **schiefe Verteilung** (hier: Power-Law).

In [12]:
def beschreibe(werte, name):
    """Gibt die wichtigsten Kennzahlen einer Zahlenliste aus."""
    print(f"--- {name} ---")
    print(f"Anzahl:              {len(werte)}")
    print(f"Summe:               {sum(werte)}")
    print(f"Minimum:             {min(werte)}")
    print(f"Maximum:             {max(werte)}")
    print(f"Mittelwert:          {mittelwert(werte):.2f}")
    print(f"Median:              {median(werte):.2f}")
    print(f"Standardabweichung:  {standardabweichung(werte):.2f}")

beschreibe(verkaufszahlen, "Verkäufe je Buch und Jahr")

--- Verkäufe je Buch und Jahr ---
Anzahl:              180
Summe:               2760
Minimum:             5
Maximum:             151
Mittelwert:          15.33
Median:              8.00
Standardabweichung:  23.32


## Übung: Kennzahlen je Jahr 💪

:::{exercise} Statistik pro Verkaufsjahr
:label: statistik-pro-jahr

Berechnen Sie für **jedes** Verkaufsjahr (`sales_year`) separat den Mittelwert
und den Median der Verkäufe. Speichern Sie die Ergebnisse in einem Dictionary
der Form `{jahr: {"mittelwert": ..., "median": ...}}` und geben Sie sie
sortiert nach Jahr aus.

Tipp: Gruppieren Sie zuerst die Verkaufszahlen mit `defaultdict(list)`.
:::

:::{solution} statistik-pro-jahr
:class: dropdown

```python
werte_pro_jahr = defaultdict(list)
for satz in DATEN:
    werte_pro_jahr[satz["sales_year"]].append(satz["sales"])

ergebnis = {}
for jahr, werte in werte_pro_jahr.items():
    ergebnis[jahr] = {
        "mittelwert": round(mittelwert(werte), 2),
        "median": median(werte),
    }

for jahr in sorted(ergebnis):
    werte = ergebnis[jahr]
    print(f"{jahr}: Mittelwert {werte['mittelwert']}, Median {werte['median']}")
```
:::

## Zusammenfassung

- Die wichtigsten Kennzahlen lassen sich mit wenigen Zeilen selbst
  implementieren: Anzahl (`len`), Summe (`sum`), Minimum/Maximum (`min`/`max`),
  Mittelwert, Median, Modus und Standardabweichung.
- **Gruppieren und Aggregieren** folgt immer demselben Muster: leeres
  Dictionary, über alle Datensätze laufen, Schlüssel als Gruppe verwenden.
- `defaultdict` und `Counter` machen diese Muster kurz und lesbar.
- Ein großer Abstand zwischen Mittelwert und Median weist auf eine schiefe
  Verteilung hin.